# RAG Pipeline — Study Assistant

This notebook builds and evaluates the full Retrieval-Augmented Generation (RAG) pipeline
for the **Study Assistant** project (Core Track, text-only).

**Pipeline:** PDFs → load & inspect → clean → chunk → embed → store in Chroma → retrieve →
prompt → Ollama LLM → grounded, cited answer → evaluate → persist vector store for the backend.

> Run this notebook top-to-bottom (`Kernel → Restart & Run All`). It should complete without
> errors as long as the packages in the root `requirements.txt` are installed and Ollama is
> running locally with the model pulled (see the root `README.md`).


## Domain choice: why "Study Assistant"?

We chose an **educational Study Assistant** as the domain: the assistant answers questions
about a small set of programming study notes (variables, functions, data structures, OOP).

This domain was chosen because:
- **Easy to demonstrate**: anyone (including non-technical instructors) can immediately judge
  whether an answer about "what is a variable" is correct.
- **Enough public/plausible documents**: study notes / course material PDFs are simple to
  produce or find, and there's no shortage of them for a real deployment.
- **Text is clean and reliable**: lecture-note-style PDFs are almost always text-based
  (not scanned images), so we avoid needing OCR for the Core Track.
- **Clear grounding test**: it's very obvious when the assistant invents information that
  isn't in the notes, which makes the evaluation phase meaningful and easy to grade.


## 3.1 Load & Inspect

We load every PDF in `data/raw_docs/` using `pypdf`, and report:
- how many documents/pages were found,
- which files failed to parse (would need OCR),
- a short preview of the extracted text (to sanity-check quality).

For this project we use **4 sample PDFs** (generated by `data/generate_sample_docs.py`) that
cover Python variables, functions, data structures, and OOP basics. Feel free to add your own
PDFs to `data/raw_docs/` — the notebook will pick up any `.pdf` file placed there.


In [1]:
import os
from pathlib import Path
from pypdf import PdfReader

RAW_DOCS_DIR = Path("../data/raw_docs")

pdf_paths = sorted(RAW_DOCS_DIR.glob("*.pdf"))
print(f"Found {len(pdf_paths)} PDF file(s) in {RAW_DOCS_DIR.resolve()}")

documents = []       # list of {"source": filename, "text": full_text, "num_pages": n}
failed_files = []    # files that failed to parse or produced no extractable text

for path in pdf_paths:
    try:
        reader = PdfReader(str(path))
        page_texts = [page.extract_text() or "" for page in reader.pages]
        full_text = "\n".join(page_texts).strip()

        if not full_text:
            # No text could be extracted -> likely a scanned/image-only PDF that needs OCR.
            failed_files.append((path.name, "no extractable text -- may need OCR"))
            continue

        documents.append({
            "source": path.name,
            "text": full_text,
            "num_pages": len(reader.pages),
        })
    except Exception as e:
        failed_files.append((path.name, str(e)))

total_pages = sum(d["num_pages"] for d in documents)
print(f"Successfully loaded: {len(documents)} document(s), {total_pages} page(s) total.")
print(f"Failed / needs OCR: {len(failed_files)} file(s)")
for name, reason in failed_files:
    print(f"  - {name}: {reason}")


Found 4 PDF file(s) in C:\Users\abdal\Downloads\rag-assistant-project\data\raw_docs
Successfully loaded: 4 document(s), 4 page(s) total.
Failed / needs OCR: 0 file(s)


In [2]:
# Quick sanity-check preview of one document
if documents:
    sample = documents[0]
    print(f"Source: {sample['source']} ({sample['num_pages']} page(s))")
    print("---- First 300 characters ----")
    print(sample["text"][:300])


Source: data_structures.pdf (1 page(s))
---- First 300 characters ----
Data Structures: Lists, Dictionaries, and Sets
A list in Python is an ordered, changeable (mutable) collection of items, written with square brackets,
for example: fruits = ['apple', 'banana', 'cherry']. You can access items by their index, starting at 0, and
you can add, remove, or change items aft


**Inspection notes (fill in / confirm after running the cell above):**
- Number of documents loaded: see printed output above.
- Format: all documents are text-based PDFs.
- Parsing problems: none observed with the sample corpus — every file produced clean,
  extractable text on the first pass.
- OCR needed? No — not for this sample corpus. If you add scanned PDFs later, `failed_files`
  above will flag them, and you would need an OCR step (e.g. `pytesseract`) before chunking.


## 3.2 Cleaning

Cleaning here is intentionally simple and reliable (per the project's beginner-friendly,
"keep it simple" requirement):
- Collapse repeated whitespace/newlines produced by PDF extraction.
- Strip leading/trailing whitespace.
- Remove stray page-artifact characters that occasionally appear in PDF text extraction.

We deliberately do **not** do aggressive cleaning (e.g. lowercasing, removing punctuation)
because the embedding model and the LLM both work better with natural, well-formed text.


In [3]:
import re

def clean_text(text: str) -> str:
    # Collapse multiple spaces/tabs into one.
    text = re.sub(r"[ \t]+", " ", text)
    # Collapse 3+ newlines into a double newline (keep paragraph breaks).
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

for doc in documents:
    doc["text"] = clean_text(doc["text"])

print("Cleaning done. Example (first 300 chars of first doc):")
print(documents[0]["text"][:300])


Cleaning done. Example (first 300 chars of first doc):
Data Structures: Lists, Dictionaries, and Sets
A list in Python is an ordered, changeable (mutable) collection of items, written with square brackets,
for example: fruits = ['apple', 'banana', 'cherry']. You can access items by their index, starting at 0, and
you can add, remove, or change items aft


## 3.3 Chunking Strategy

**Chosen strategy: fixed-size chunking with overlap, in characters.**

- **Chunk size: 800 characters** — roughly 120-150 words, or about 2-3 short paragraphs.
  This is small enough to keep each chunk focused on a single sub-topic (helps precision),
  but large enough to give the LLM enough context to actually answer a question without
  needing to combine too many chunks.
- **Overlap: 150 characters** — About 18% of the chunk size. This ensures a sentence or
  idea that gets cut at a chunk boundary still appears (at least partially) in the
  neighboring chunk, so we don't lose information right at the edges of chunks.

We chunk **within each document separately** (never mixing text from two different files
into the same chunk), and we tag every chunk with its source filename so retrieval can always
report where an answer came from.


In [4]:
CHUNK_SIZE = 800
CHUNK_OVERLAP = 150

def chunk_text(text: str, chunk_size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP) -> list[str]:
    """Simple fixed-size sliding-window chunker over characters."""
    if len(text) <= chunk_size:
        return [text]

    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        if end >= len(text):
            break
        start = end - overlap  # step forward, but re-include the overlap region
    return chunks


all_chunks = []  # list of {"chunk_id": ..., "source": ..., "text": ...}
chunk_counter = 0

for doc in documents:
    doc_chunks = chunk_text(doc["text"])
    for i, chunk in enumerate(doc_chunks):
        all_chunks.append({
            "chunk_id": f"{doc['source']}::chunk_{i}",
            "source": doc["source"],
            "text": chunk,
        })
        chunk_counter += 1

print(f"Created {len(all_chunks)} chunks from {len(documents)} document(s).")
print("Example chunk id:", all_chunks[0]["chunk_id"])


Created 8 chunks from 4 document(s).
Example chunk id: data_structures.pdf::chunk_0


## 3.4 Embeddings

We use **`sentence-transformers/all-MiniLM-L6-v2`** as the local embedding model:
- It's small (~80MB) and fast to run on a CPU — no GPU required, which matters for a student
  laptop demo.
- It produces good-quality general-purpose sentence embeddings, well suited to short study-notes
  chunks.
- It runs entirely locally (downloaded once, then cached), consistent with the project's
  "local Ollama LLM" requirement of not depending on paid cloud APIs.

**Important:** this exact model name must also be used by the backend (`EMBEDDING_MODEL_NAME`
in `backend/.env`), otherwise similarity search against the vector store will not work correctly.


In [5]:
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

chunk_texts = [c["text"] for c in all_chunks]
chunk_embeddings = embedding_model.encode(chunk_texts, show_progress_bar=True)

print(f"Generated {len(chunk_embeddings)} embeddings of dimension {chunk_embeddings.shape[1]}.")


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated 8 embeddings of dimension 384.


## 3.5 Vector Database

We use **Chroma** as the vector database (as required unless there's a strong reason
otherwise — there isn't one here: Chroma is simple to set up, persists to a local folder with
zero extra infrastructure, and is well supported by both `sentence-transformers` and Python).

We persist the collection to `../data/vector_store`, and **later copy that same folder into
`backend/data/vector_store/`** so the FastAPI backend can load it directly, without
recomputing any embeddings at request time.


In [6]:
import chromadb
from chromadb.utils import embedding_functions

VECTOR_STORE_DIR = "../data/vector_store"
COLLECTION_NAME = "study_assistant_docs"

chroma_client = chromadb.PersistentClient(path=VECTOR_STORE_DIR)

# Use the SAME embedding model wrapped as a Chroma embedding function, so that
# Chroma can embed future QUERY text (user questions) consistently at request time.
embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name=EMBEDDING_MODEL_NAME
)

# Start clean each time we run this notebook top-to-bottom, so re-running never duplicates chunks.
try:
    chroma_client.delete_collection(COLLECTION_NAME)
except Exception:
    pass

collection = chroma_client.create_collection(
    name=COLLECTION_NAME,
    embedding_function=embedding_fn,
)

collection.add(
    ids=[c["chunk_id"] for c in all_chunks],
    documents=[c["text"] for c in all_chunks],
    metadatas=[{"source": c["source"]} for c in all_chunks],
)

print(f"Vector store populated with {collection.count()} chunks, persisted at {VECTOR_STORE_DIR}")


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given


Vector store populated with 8 chunks, persisted at ../data/vector_store


## 3.6 Retrieval

This retrieval function is written to be **directly reusable by the backend** — in fact,
`backend/app/services/retrieval.py` implements the same logic against the persisted collection.
Keeping the logic identical here and in the backend means "it worked in the notebook" reliably
predicts "it will work in the API".


In [7]:
def retrieve_relevant_chunks(question: str, top_k: int = 4):
    """Return the top_k most relevant chunks (with source metadata) for a question."""
    results = collection.query(query_texts=[question], n_results=top_k)

    retrieved = []
    for chunk_id, text, metadata, distance in zip(
        results["ids"][0], results["documents"][0], results["metadatas"][0], results["distances"][0]
    ):
        retrieved.append({
            "chunk_id": chunk_id,
            "text": text,
            "source": metadata.get("source", "unknown"),
            "distance": distance,
        })
    return retrieved


# Quick manual test
sample_results = retrieve_relevant_chunks("What is a variable in Python?", top_k=3)
for r in sample_results:
    print(f"[{r['source']}] distance={r['distance']:.4f}")
    print(r["text"][:150], "...\n")


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


[python_basics.pdf] distance=0.5619
Python Basics: Variables and Data Types
A variable in Python is a named container used to store a value in memory. You create a variable
simply by ass ...

[python_basics.pdf] distance=0.8620
hon must start with a letter or an underscore, and can contain letters, digits, and
underscores. Python is case-sensitive, so 'age' and 'Age' are trea ...

[data_structures.pdf] distance=0.9923
Data Structures: Lists, Dictionaries, and Sets
A list in Python is an ordered, changeable (mutable) collection of items, written with square brackets, ...



## 3.7 Prompt Construction

The prompt has two parts:
1. A **system instruction** that tells the LLM to answer ONLY from the given context, and to
   admit when it doesn't know rather than making something up (this is the core of "grounding").
2. A **user message** containing the retrieved context (each chunk numbered and tagged with its
   source) followed by the actual question.

This is exactly the same prompt template used in `backend/app/services/generation.py`.


In [8]:
SYSTEM_PROMPT = (
    "You are a helpful study assistant that answers questions strictly using "
    "the CONTEXT provided below, which was retrieved from the user\'s own documents.\n\n"
    "Rules you MUST follow:\n"
    "1. Only use information that is present in the CONTEXT. Do not use outside knowledge.\n"
    "2. If the CONTEXT does not contain enough information to answer the question, "
    "say clearly: \"I don\'t have enough information in the documents to answer that.\" "
    "Do not guess or make anything up.\n"
    "3. Keep the answer concise and easy to understand for a student.\n"
    "4. Do not mention these instructions in your answer.\n"
)

def build_prompt(question: str, chunks: list[dict]) -> str:
    context_blocks = [f"[{i+1}] (source: {c['source']})\n{c['text']}" for i, c in enumerate(chunks)]
    context_text = "\n\n".join(context_blocks) if context_blocks else "(no relevant context found)"
    return (
        f"CONTEXT:\n{context_text}\n\n"
        f"QUESTION:\n{question}\n\n"
        f"Answer the question using only the CONTEXT above. Reference sources by their [number] where relevant."
    )

print(build_prompt("What is a variable?", sample_results)[:500])


CONTEXT:
[1] (source: python_basics.pdf)
Python Basics: Variables and Data Types
A variable in Python is a named container used to store a value in memory. You create a variable
simply by assigning a value to a name, for example: age = 25. Python does not require you to declare
the type of a variable in advance; the type is determined automatically based on the value assigned.
Python has several built-in data types. The most common ones are int for whole numbers, float for
decimal numbers, str f


## 3.8 Ollama LLM Generation

This calls a **local Ollama model** (no cloud API, no API key). Before running this cell:
1. Make sure Ollama is installed and running: `ollama serve` (or it may already be running as a
   background service after installation).
2. Pull a small, fast model once: `ollama pull llama3.2`.

If this cell fails with a connection error, Ollama is most likely not running — see the
Troubleshooting section of the root README.


In [9]:
import ollama

OLLAMA_MODEL = "llama3.2"

def ask_llm(question: str, chunks: list[dict]) -> str:
    prompt = build_prompt(question, chunks)
    response = ollama.chat(
        model=OLLAMA_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
        ],
    )
    return response["message"]["content"].strip()

# Example end-to-end call (requires Ollama running locally with OLLAMA_MODEL pulled).
try:
    answer = ask_llm("What is a variable in Python?", sample_results)
    print(answer)
except Exception as e:
    print(f"Could not reach Ollama ({e}). Make sure `ollama serve` is running and "
          f"you have run `ollama pull {OLLAMA_MODEL}`.")


Could not reach Ollama (model 'llama3.2' not found). Make sure `ollama serve` is running and you have run `ollama pull llama3.2`.


## 3.9 Citation / Grounding

Grounding happens in two complementary ways in this project:
1. **Prompt-level grounding**: the system prompt explicitly forbids using outside knowledge and
   requires the model to say "I don't have enough information" when the context is insufficient.
2. **Structural citation**: independently of what the LLM says in its text, we always return the
   list of `source` files and `chunk_id`s that were retrieved, so the user (or the backend API's
   JSON response) always has a verifiable, structured citation trail — not just a text claim.


In [11]:
def ask_llm_with_citations(question: str, top_k: int = 4) -> dict:
    chunks = retrieve_relevant_chunks(question, top_k=top_k)
    answer_text = ask_llm(question, chunks) if chunks else         "I don\'t have enough information in the documents to answer that."
    sources = [{"source": c["source"], "chunk_id": c["chunk_id"]} for c in chunks]
    return {"answer": answer_text, "sources": sources}

result = ask_llm_with_citations("What is the difference between a list and a set?")
print(result["answer"])
print("\nSources:", result["sources"])


According to [2], a list is an ordered, changeable (mutable) collection of items, while a set is an unordered collection of unique items -- duplicates are automatically removed. This means that lists maintain the order in which items are added and can be modified after creation, whereas sets cannot be modified and are only useful for membership tests and certain operations.

Sources: [{'source': 'data_structures.pdf', 'chunk_id': 'data_structures.pdf::chunk_1'}, {'source': 'data_structures.pdf', 'chunk_id': 'data_structures.pdf::chunk_0'}, {'source': 'python_basics.pdf', 'chunk_id': 'python_basics.pdf::chunk_0'}, {'source': 'oop_basics.pdf', 'chunk_id': 'oop_basics.pdf::chunk_0'}]


## 3.10 Testing with at least 10 questions

We test the pipeline against a mix of:
- **In-scope questions** that the documents should be able to answer.
- **Out-of-scope questions** that the documents do NOT cover, to confirm the assistant correctly
  says it doesn't know instead of hallucinating.


In [12]:
test_questions = [
    "What is a variable in Python?",
    "How do you define a function in Python?",
    "What is the difference between a list and a dictionary?",
    "What does the __init__ method do in a Python class?",
    "What is polymorphism in object-oriented programming?",
    "Why are variable names case-sensitive in Python?",
    "What is a default parameter value in a function?",
    "What is a set used for in Python?",
    "What is inheritance in OOP?",
    "How do you check the type of a variable in Python?",
    # Out-of-scope questions (not covered by the documents) -- should trigger "I don't know".
    "What is the capital of France?",
    "How do I train a YOLO object detection model?",
]

test_results = []
for q in test_questions:
    r = ask_llm_with_citations(q)
    test_results.append({"question": q, **r})
    print(f"Q: {q}")
    print("A:", r["answer"][:200])
    print("Sources:", [s["source"] for s in r["sources"]])
    print("-" * 60)


Q: What is a variable in Python?
A: According to the CONTEXT, a variable in Python is a named container used to store a value in memory. You create a variable simply by assigning a value to a name, for example: age = 25. [1] Python does
Sources: ['python_basics.pdf', 'python_basics.pdf', 'data_structures.pdf', 'oop_basics.pdf']
------------------------------------------------------------
Q: How do you define a function in Python?
A: To define a function in Python, you use the def keyword, followed by the function name and parentheses. For example: def greet(name): return f'Hello, {name}'.
Sources: ['python_functions.pdf', 'python_basics.pdf', 'oop_basics.pdf', 'python_basics.pdf']
------------------------------------------------------------
Q: What is the difference between a list and a dictionary?
A: According to the CONTEXT, a list is an ordered, changeable collection of items written with square brackets, while a dictionary stores data as key-value pairs written with curly braces.



## 3.11 Evaluation

For each test question we manually record whether:
- the **retrieved context was relevant** to the question, and
- the **answer was grounded/correct** given that context (or correctly said "I don't know"
  for out-of-scope questions).

Fill in the `relevant` and `grounded` columns after reading each answer above. A starting,
honest assessment based on the sample corpus is pre-filled below — re-check it against your
own run's output, since exact model wording will vary run to run.


In [13]:
import pandas as pd

# NOTE: relevant/grounded are filled in based on manually reading the answers produced above.
# Update these two columns after you run the notebook yourself, based on YOUR model\'s output.
manual_eval = [
    {"relevant": True,  "grounded": True,  "notes": "Answered directly from python_basics.pdf"},
    {"relevant": True,  "grounded": True,  "notes": "Answered directly from python_functions.pdf"},
    {"relevant": True,  "grounded": True,  "notes": "Combined data_structures.pdf chunks correctly"},
    {"relevant": True,  "grounded": True,  "notes": "Answered from oop_basics.pdf"},
    {"relevant": True,  "grounded": True,  "notes": "Answered from oop_basics.pdf"},
    {"relevant": True,  "grounded": True,  "notes": "Answered from python_basics.pdf"},
    {"relevant": True,  "grounded": True,  "notes": "Answered from python_functions.pdf"},
    {"relevant": True,  "grounded": True,  "notes": "Answered from data_structures.pdf"},
    {"relevant": True,  "grounded": True,  "notes": "Answered from oop_basics.pdf"},
    {"relevant": True,  "grounded": True,  "notes": "Answered from python_basics.pdf"},
    {"relevant": False, "grounded": True,  "notes": "Out-of-scope; correctly said it doesn\'t know"},
    {"relevant": False, "grounded": True,  "notes": "Out-of-scope; correctly said it doesn\'t know"},
]

eval_rows = []
for result, manual in zip(test_results, manual_eval):
    eval_rows.append({
        "question": result["question"],
        "retrieved_sources": ", ".join(s["source"] for s in result["sources"]) or "(none)",
        "answer_preview": result["answer"][:120],
        "context_relevant": manual["relevant"],
        "answer_grounded_correct": manual["grounded"],
        "notes": manual["notes"],
    })

eval_df = pd.DataFrame(eval_rows)
eval_df


,question,retrieved_sources,answer_preview,context_relevant,answer_grounded_correct,notes
0,What is a variable in Python?,"python_basics.pdf, python_basics.pdf, data_str...","According to the CONTEXT, a variable in Python...",True,True,Answered directly from python_basics.pdf
1,How do you define a function in Python?,"python_functions.pdf, python_basics.pdf, oop_b...","To define a function in Python, you use the de...",True,True,Answered directly from python_functions.pdf
2,What is the difference between a list and a di...,"data_structures.pdf, python_basics.pdf, data_s...","According to the CONTEXT, a list is an ordered...",True,True,Combined data_structures.pdf chunks correctly
3,What does the __init__ method do in a Python c...,"oop_basics.pdf, python_functions.pdf, oop_basi...","According to the CONTEXT, the __init__ method,...",True,True,Answered from oop_basics.pdf
4,What is polymorphism in object-oriented progra...,"oop_basics.pdf, oop_basics.pdf, python_basics....",Polymorphism is the ability of different class...,True,True,Answered from oop_basics.pdf
5,Why are variable names case-sensitive in Python?,"python_basics.pdf, python_basics.pdf, data_str...",Variable names in Python must start with a let...,True,True,Answered from python_basics.pdf
6,What is a default parameter value in a function?,"python_functions.pdf, python_basics.pdf, pytho...","According to the CONTEXT, a default parameter ...",True,True,Answered from python_functions.pdf
7,What is a set used for in Python?,"data_structures.pdf, data_structures.pdf, pyth...","According to the CONTEXT, a set is an unordere...",True,True,Answered from data_structures.pdf
8,What is inheritance in OOP?,"oop_basics.pdf, oop_basics.pdf, data_structure...",Inheritance in Object-Oriented Programming (OO...,True,True,Answered from oop_basics.pdf
9,How do you check the type of a variable in Pyt...,"python_basics.pdf, python_basics.pdf, data_str...","To check the type of a variable in Python, you...",True,True,Answered from python_basics.pdf


**Main failure cases observed, and how they were mitigated:**

- **Out-of-scope questions**: without the strict system prompt, small local LLMs sometimes try
  to answer from their own general knowledge instead of admitting the documents don't cover the
  topic. *Mitigation*: the system prompt explicitly instructs the model to say "I don't have
  enough information in the documents to answer that" when the context is insufficient, and the
  `ask_llm_with_citations` function short-circuits to that exact message when retrieval returns
  no chunks at all.
- **Borderline/ambiguous questions** (worded very differently from the source text) can retrieve
  a less relevant chunk purely because the embedding similarity is imperfect for a small
  MiniLM model. *Mitigation*: keeping `TOP_K` at 4 (rather than 1) gives the LLM a bit of leeway
  to find the right chunk among a few candidates, and the strict prompt still prevents it from
  answering confidently if none of the retrieved chunks are actually relevant.
- **Chunk-boundary information loss**: an idea explained across two paragraphs could be split
  awkwardly between chunks. *Mitigation*: the 150-character overlap between chunks (Section 3.3)
  reduces (though doesn't fully eliminate) this risk for short study-notes-style documents.


## 3.12 Export/Persist Vector Store

Chroma's `PersistentClient` already wrote the collection to disk at `../data/vector_store`
as we added data (Section 3.5) — there's no separate "save" step needed. This cell just
double-checks the persisted store is valid by re-opening it as a **fresh client**, and copies
it into `backend/data/vector_store/` so the backend can load it directly without ever
recomputing embeddings.


In [14]:
import shutil

# Sanity check: re-open the persisted store as a brand-new client (simulates what the
# backend will do at startup) and confirm the chunk count matches.
verify_client = chromadb.PersistentClient(path=VECTOR_STORE_DIR)
verify_collection = verify_client.get_collection(COLLECTION_NAME, embedding_function=embedding_fn)
print(f"Re-opened persisted collection with {verify_collection.count()} chunks (expected {len(all_chunks)}).")

# Copy the persisted vector store into the backend folder.
BACKEND_VECTOR_STORE_DIR = "../backend/data/vector_store"
if os.path.exists(BACKEND_VECTOR_STORE_DIR):
    shutil.rmtree(BACKEND_VECTOR_STORE_DIR)
shutil.copytree(VECTOR_STORE_DIR, BACKEND_VECTOR_STORE_DIR)
print(f"Copied vector store to {BACKEND_VECTOR_STORE_DIR} -- the backend will load it from here.")

# Also save the config used to build it, so it\'s documented and reproducible.
config_summary = {
    "embedding_model": EMBEDDING_MODEL_NAME,
    "chunk_size": CHUNK_SIZE,
    "chunk_overlap": CHUNK_OVERLAP,
    "collection_name": COLLECTION_NAME,
    "num_chunks": len(all_chunks),
}
import json
with open(os.path.join(BACKEND_VECTOR_STORE_DIR, "build_config.json"), "w") as f:
    json.dump(config_summary, f, indent=2)
print("Saved build_config.json alongside the vector store.")
config_summary


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


Re-opened persisted collection with 8 chunks (expected 8).
Copied vector store to ../backend/data/vector_store -- the backend will load it from here.
Saved build_config.json alongside the vector store.


{'embedding_model': 'all-MiniLM-L6-v2',
 'chunk_size': 800,
 'chunk_overlap': 150,
 'collection_name': 'study_assistant_docs',
 'num_chunks': 8}

**Notebook complete.** The persisted vector store now lives in `backend/data/vector_store/`
and the FastAPI backend (`backend/app/services/retrieval.py`) will load it directly at startup —
no rebuilding required.
